# 03 - Preprocessing

This notebook handles the preprocessing pipeline with strict leakage prevention:
1. Load merged dataset
2. Train/Test split (BEFORE any preprocessing)
3. Build preprocessing pipelines (Clinical + RNA)
4. Fit pipelines ONLY on training data
5. Transform both train and test data
6. Save preprocessed datasets

> **Critical Rule:** All transformers are fitted on training data ONLY.
> Test data is never seen during fitting.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import config
from src.io import save_dataframe, logger
from src.preprocessing import (
    build_clinical_pipeline,
    build_rna_pipeline,
    build_combined_pipeline,
    identify_column_groups,
    fit_preprocessing,
    transform_data,
)
from src.leakage import drop_leakage_columns

## Step 1: Load Merged Dataset

In [2]:
X = pd.read_csv(config.PROCESSED_DIR / "X_features_final.csv")
y = pd.read_csv(config.PROCESSED_DIR / "y_target_final.csv").iloc[:, 0]

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Positive rate: {y.mean():.2%}")

Dataset shape: (429, 19019)
Target distribution:
Biochemical_Recurrence_Code
0    371
1     58
Name: count, dtype: int64
Positive rate: 13.52%


## Step 2: Train/Test Split (BEFORE any preprocessing)

> **Leakage Prevention:** The split happens BEFORE any preprocessing
> to ensure test data is never seen during fitting.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    stratify=y,
    random_state=config.RANDOM_STATE,
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"Train shape: {X_train.shape}, positives={int(y_train.sum())}")
print(f"Test shape:  {X_test.shape}, positives={int(y_test.sum())}")
print(f"Train positive rate: {y_train.mean():.2%}")
print(f"Test positive rate:  {y_test.mean():.2%}")

Train shape: (343, 19019), positives=46
Test shape:  (86, 19019), positives=12
Train positive rate: 13.41%
Test positive rate:  13.95%


## Step 3: Identify Column Groups (Clinical vs Gene)

In [4]:
clinical_cols, gene_cols = identify_column_groups(X_train)

print(f"Clinical features: {len(clinical_cols)}")
print(f"Gene features:     {len(gene_cols)}")
print(f"Total features:    {len(clinical_cols) + len(gene_cols)}")
print(f"\nSample clinical features: {clinical_cols[:5]}")
print(f"Sample gene features:     {gene_cols[:5]}")

2026-08-25 23:01:50 | INFO     | prostate_bcr | Column groups: 183 clinical, 18836 gene


Clinical features: 183
Gene features:     18836
Total features:    19019

Sample clinical features: ['Gleason pattern primary', 'Gleason pattern secondary', 'Year Cancer Initial Diagnosis', 'Lymph Node(s) Examined Number', 'Diagnosis Age']
Sample gene features:     ['Radical Prostatectomy Gleason Score for Prostate Cancer', 'Positive Finding Lymph Node Hematoxylin and Eosin Staining Microscopy Count', 'Prior Cancer Diagnosis Occurence_No', 'Prior Cancer Diagnosis Occurence_Yes', 'Prior Cancer Diagnosis Occurence_Yes, History of Prior Malignancy']


## Step 4: Build Combined Preprocessing Pipeline

The pipeline applies different transformations to clinical and gene features:
- **Clinical:** Imputation → Log1p (skewed) → Winsorize → StandardScaler
- **Genes:** Imputation → Log2(x+1) → StandardScaler

In [5]:
preprocessor = build_combined_pipeline(
    X_train,
    apply_log_skew=True,
    apply_winsorize=True,
)

print("Combined pipeline built successfully.")
print(f"Transformers: {preprocessor.transformers}")

2026-08-25 23:01:50 | INFO     | prostate_bcr | Column groups: 183 clinical, 18836 gene


Combined pipeline built successfully.
Transformers: [('clinical', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('log_skew', Log1pSkewTransformer()),
                ('winsorize', WinsorizeTransformer()),
                ('scaler', StandardScaler())]), ['Gleason pattern primary', 'Gleason pattern secondary', 'Year Cancer Initial Diagnosis', 'Lymph Node(s) Examined Number', 'Diagnosis Age', 'Tumor Other Histologic Subtype_25-30% ductal component', 'Tumor Other Histologic Subtype_Adenocarcinoma prostate with prominent ductal differentiation identified', 'Tumor Other Histologic Subtype_Mixed', 'Tumor Other Histologic Subtype_Mixed ductal (65%) and Acinar', 'Tumor Other Histologic Subtype_Prostate Adenocarcinoma, Not Otherwised Specified, with ductal featues', 'Tumor Other Histologic Subtype_Prostatic Adenocarcinoma Acinar Type Mixed with Prostatic Duct Adenocarcinoma', 'Tumor Other Histologic Subtype_acinar and cribriform', 'Tumor Other Histologic Subtype_

## Step 5: Fit Pipeline on Training Data ONLY

> **Leakage Prevention:** The pipeline is fitted ONLY on `X_train`.
> Test data (`X_test`) is never used during fitting.

In [6]:
preprocessor_fitted = fit_preprocessing(preprocessor, X_train, y_train)
print("Pipeline fitted on training data successfully.")

2026-08-25 23:01:53 | INFO     | prostate_bcr | Preprocessing fitted on 343 training samples


Pipeline fitted on training data successfully.


## Step 6: Transform Both Train and Test Data

In [7]:
X_train_processed = transform_data(preprocessor_fitted, X_train)
X_test_processed = transform_data(preprocessor_fitted, X_test)

print(f"Train processed shape: {X_train_processed.shape}")
print(f"Test processed shape:  {X_test_processed.shape}")
print(f"\nTrain statistics:")
print(f"  Mean:  {X_train_processed.mean():.4f}")
print(f"  Std:   {X_train_processed.std():.4f}")
print(f"  Min:   {X_train_processed.min():.4f}")
print(f"  Max:   {X_train_processed.max():.4f}")

Train processed shape: (343, 19019)
Test processed shape:  (86, 19019)

Train statistics:
  Mean:  0.0000
  Std:   0.9991
  Min:   -12.7433
  Max:   18.4932


## Step 7: Verify Preprocessing Quality

In [8]:
# Check for any remaining NaN or Inf values
print("Quality checks after preprocessing:")
print(f"  Train NaN count: {np.isnan(X_train_processed).sum()}")
print(f"  Test NaN count:  {np.isnan(X_test_processed).sum()}")
print(f"  Train Inf count: {np.isinf(X_train_processed).sum()}")
print(f"  Test Inf count:  {np.isinf(X_test_processed).sum()}")

# Verify scaling (mean should be ~0, std should be ~1)
print(f"\nScaling verification:")
print(f"  Train mean (should be ~0): {X_train_processed.mean():.6f}")
print(f"  Train std  (should be ~1): {X_train_processed.std():.6f}")

Quality checks after preprocessing:
  Train NaN count: 0
  Test NaN count:  0
  Train Inf count: 0
  Test Inf count:  0

Scaling verification:
  Train mean (should be ~0): 0.000000
  Train std  (should be ~1): 0.999106


## Step 8: Save Preprocessed Data

In [9]:
# Save as DataFrames with proper column names
feature_names = X_train.columns.tolist()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

save_dataframe(X_train_df, "X_train_preprocessed.csv", index=False)
save_dataframe(X_test_df, "X_test_preprocessed.csv", index=False)
save_dataframe(y_train.to_frame(), "y_train.csv", index=False)
save_dataframe(y_test.to_frame(), "y_test.csv", index=False)

print("Preprocessed data saved successfully:")
print(f"  - X_train_preprocessed.csv ({X_train_df.shape})")
print(f"  - X_test_preprocessed.csv ({X_test_df.shape})")
print(f"  - y_train.csv ({y_train.shape})")
print(f"  - y_test.csv ({y_test.shape})")

2026-08-25 23:02:03 | INFO     | prostate_bcr | Saved 343 rows → D:\Prostate_BCR\core\data\processed\X_train_preprocessed.csv
2026-08-25 23:02:06 | INFO     | prostate_bcr | Saved 86 rows → D:\Prostate_BCR\core\data\processed\X_test_preprocessed.csv
2026-08-25 23:02:06 | INFO     | prostate_bcr | Saved 343 rows → D:\Prostate_BCR\core\data\processed\y_train.csv
2026-08-25 23:02:06 | INFO     | prostate_bcr | Saved 86 rows → D:\Prostate_BCR\core\data\processed\y_test.csv


Preprocessed data saved successfully:
  - X_train_preprocessed.csv ((343, 19019))
  - X_test_preprocessed.csv ((86, 19019))
  - y_train.csv ((343,))
  - y_test.csv ((86,))
